## 0. Harness patch

Patch to add our modifications to the original tufa harness.

In [ ]:
# Load and verify the versioned agent source; no runtime agent patching.
import hashlib
import json
import sys
from pathlib import Path

if "inference.agent.python_tool_sandbox" in sys.modules:
    raise RuntimeError("Start a fresh Kaggle session before switching source packages.")
EXPECTED_MANIFEST_SHA256 = '8070f6dd3aa76861920e93dbdd3a8f4ed9b9d0744cf0360db80db301670b028d'
_candidates = [p for p in Path("/kaggle/input").rglob("dhia-m2-manifest.json")
               if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED_MANIFEST_SHA256]
if len(_candidates) != 1:
    raise RuntimeError("Attach exactly one extracted dhia-m2-source-0.1.21 dataset; expected manifest not uniquely found.")
SOURCE_BUNDLE_DIR = _candidates[0].parent
_manifest = json.loads(_candidates[0].read_text())
for _name, _digest in _manifest["files"].items():
    _file = (SOURCE_BUNDLE_DIR / _name).resolve()
    if not _file.is_relative_to(SOURCE_BUNDLE_DIR.resolve()):
        raise RuntimeError("Unsafe source manifest path")
    if not _file.is_file() or hashlib.sha256(_file.read_bytes()).hexdigest() != _digest:
        raise RuntimeError("Source integrity failure: " + _name)
print("Verified source:", _manifest["release"], "at", SOURCE_BUNDLE_DIR)


## 1. Environment and submission mode

Detect whether this is a real competition rerun (which minimises diagnostics), set the
framework's environment flags, and put the CUDA libraries on the linker path.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# Kaggle might cancel the run in no action is done in some timeframe.
# To prevent this, harness will start up even if backend is not yet ready,
# and the first started game will perform a single reset action.
NOTEBOOK_START_TIME = time.time()
SERVER_STARTUP_TIMEOUT = 12 * 60
os.environ['ARC3_WARMUP_ACTION_GAMES'] = '1'
os.environ['ARC3_HTTP_RETRY_INITIAL_SECONDS'] = '900'

# model and dataset paths (Duck-27B: public driessmit1 datasets, probed mounts)
def _resolve_dataset(owner, slug):
    for cand in (Path('/kaggle/input') / slug,
                 Path('/kaggle/input/datasets') / owner / slug):
        if cand.exists():
            return str(cand)
    return str(Path('/kaggle/input') / slug)  # serving cell raises a clear error

WHEELHOUSE_DIR    = _resolve_dataset('driessmit1', 'arc3-vllm-h100-wheelhouse-v3')
MODEL_DIR         = _resolve_dataset('driessmit1', 'qwen3-8-27b-fp8-hf-017b9c7a')
ORIG_BUNDLE_DIR   = str(SOURCE_BUNDLE_DIR)
SERVED_MODEL_NAME = 'Qwen/Qwen3.8-27B-FP8'

# Copy the verified, prepatched source package
"""Launcher-only copy helper. Never imports or modifies agent code."""
from pathlib import Path
import shutil


def copy_verified_bundle(source, destination, *, allowed_destination, input_root):
    source = Path(source).resolve()
    raw_destination = Path(destination)
    destination = raw_destination.resolve()
    allowed_destination = Path(allowed_destination).resolve()
    input_root = Path(input_root).resolve()
    if not source.is_dir():
        raise RuntimeError('Verified source directory does not exist')
    # All validation precedes deletion or copying.
    if raw_destination.is_symlink():
        raise RuntimeError('Bundle destination must not be a symlink')
    if destination == input_root or input_root in destination.parents:
        raise RuntimeError('Bundle destination must not be inside read-only inputs')
    if destination != allowed_destination:
        raise RuntimeError('Bundle destination is not the designated writable copy path')
    if source == destination or source in destination.parents or destination in source.parents:
        raise RuntimeError('Source and destination must not overlap')
    if destination.exists() and not destination.is_dir():
        raise RuntimeError('Bundle destination exists but is not a directory')
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(source, destination)
    return destination

# Input is read-only; BUNDLE_DIR is the designated writable copy.
BUNDLE_DIR = Path('/kaggle/taaf-kaggle-source-share')
copy_verified_bundle(
    ORIG_BUNDLE_DIR, BUNDLE_DIR,
    allowed_destination='/kaggle/taaf-kaggle-source-share',
    input_root='/kaggle/input',
)
# Agent source was built and hash-verified before copying; do not apply the upstream patch twice.
print('versioned source copied successfully')

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")
SERVED_MODEL_HOST = '127.0.0.1'
SERVED_MODEL_PORT = 1234  # Duck vLLM default
SERVER_BASE_URL = f'http://{SERVED_MODEL_HOST}:{SERVED_MODEL_PORT}/v1'
USE_PRIORITY_SCHEDULING = True

setup_env = {
    ########## transformers config ##########
    'USE_TF': '0',
    'TRANSFORMERS_NO_TF': '1',
    'TRANSFORMERS_NO_TORCHVISION': '1',

    ########## serving config ##########
    'LOCAL_ANALYZER_BASE_URL': SERVER_BASE_URL,
    'OPENAI_BASE_URL': SERVER_BASE_URL,
    'LOCAL_ANALYZER_PROVIDER': 'vllm',
    'OPENAI_PROVIDER': 'vllm',
    'LOCAL_ANALYZER_MODEL_ID': SERVED_MODEL_NAME,
    'INFERENCE_ANALYZER_MODEL': SERVED_MODEL_NAME,
    'LOCAL_ANALYZER_APP_NAME': 'ARC3 Agent Harness',
    
    ########## analyzer config ##########
    'ARC3_REASONING_HISTORY_KEY': 'reasoning_content',
    'LOCAL_ANALYZER_ENABLE_THINKING': 'true',
    'LOCAL_ANALYZER_TOOL_STEPS': '0',
    'LOCAL_ANALYZER_TOOL_TIMEOUT': '30',
    'LOCAL_ANALYZER_YIELD_SECONDS': '60',
    'LOCAL_ANALYZER_YIELD_TOKENS': '0',
    'LOCAL_ANALYZER_TEMPERATURE': '0.6',
    'LOCAL_ANALYZER_TOP_P': '0.95',
    'LOCAL_ANALYZER_TOP_K': '20',

    ########## context pruning ##########
    'LOCAL_ANALYZER_CONTEXT_WINDOW': 32768,
    'LOCAL_ANALYZER_MAX_OUTPUT': 0,
    'ARC3_CONTEXT_DRAIN_TOKENS': 58*1024,
    'ARC3_HISTORY_ASSISTANT_TURNS': 150,
    'ARC3_HISTORY_TURN_DRAIN': 30,
    'ARC3_HISTORY_DRAIN_COALESCE': '1',

    ########## tool output ##########
    'LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS': '1024',
    'ARC3_MIDDLE_TRUNCATION': '1',
    'ARC3_ACTION_ECHO': '1',
    
    ########## prompt changes ##########
    'ARC3_EXPLAIN_GAMEPLAY_CHANGED': '1',
    'ARC3_NEW_CHANGED_PROMPTS': '1',
    'ARC3_LEVEL_TRANSFER_GUIDANCE': '1',
    'ARC3_ACTION_INFO': '1',
    'ARC3_DEDUPE_MULTICALL_LINE': '1',
    'ARC3_FRAME_DIFF_HINT': '1',

    ########## vision & animations ##########
    'MULTIMODAL_CONTEXT': 'current_grid',
    'MULTIMODAL_UPSCALE': '4',
    'ARC3_DIFF_IMAGE': '1',
    'ARC3_GAMEOVER_DIFF_IMAGE': '1',
    'ARC3_ANIMATION': '1',

    ########## guards ##########
    'ARC3_GUARDS_FROM_LEVEL': 2,
    'ARC3_BATCH_NOOP_BLOCK': '1',
    'ARC3_STALE_STATE_BLOCK': '1',

    ########## world model ##########
    'ARC3_MEMORY_SECTIONS': 'on',

    ########## persistent functions ##########
    'ARC3_PERSISTENT_FUNCTIONS': '1',
    'ARC3_PERSISTENT_FUNCTIONS_SCOPE': 'game',
    'ARC3_PERSISTENT_FUNCTIONS_IMPORTS': '1',

    ########## controls ##########
    'EXPOSE_UNDO': 'off',
}

if USE_PRIORITY_SCHEDULING: setup_env.update({
    'ARC3_MAX_ACTIVE_STREAMS': 10,
    'ARC3_PRIORITY_REFRESH_QUEUE': '1',
    'ARC3_PRIORITY_PACE': '0',
    'ARC3_PRIORITY_TAIL_FADE': '1',
    'ARC3_PRIORITY_TAIL_FADE_FRACTION': '0.2',
    'ARC3_PRIORITY_TAIL_LOOKUP': 'remaining',
    'ARC3_PRIORITY_SCORE_NORMALIZATION': '1',
})

os.environ.update({k: str(v) for k,v in setup_env.items()})

## 2. Precaching

Precache wheelhouse for faster startup.

In [ ]:
import os
import threading
import time

from concurrent.futures import as_completed, ThreadPoolExecutor

def precache(*paths, times=1, delay=0, threads=16) -> None:
    time.sleep(delay)
    print(f'Precaching: Started...')
    start_time = time.time()

    CHUNK_SIZE = 32 * 1024 * 1024  # 128 MB

    chunks = []  # list of (path, offset, length)
    total_size = 0
    file_count = 0

    def _add_file(i: int, path: str) -> None:
        nonlocal total_size, file_count
        size = os.path.getsize(path)
        total_size += size
        file_count += 1
        for offset in range(0, max(size, 1), CHUNK_SIZE):
            length = min(CHUNK_SIZE, size - offset)
            chunks.append((i, path, offset, length))

    for i, path in enumerate(paths):
        try:
            if os.path.isfile(path):
                _add_file(i, path)
                continue
            for root, _, files in os.walk(path):
                for file_name in files:
                    file_path = os.path.join(root, file_name)
                    if os.path.isfile(file_path):
                        _add_file(i, file_path)
        except: pass

    chunks.sort()

    _local = threading.local()

    def _read_chunk(chunk: tuple) -> None:
        try:
            buf = _local.buf
        except AttributeError:
            _local.buf = buf = bytearray(CHUNK_SIZE)
        i, path, offset, _ = chunk
        try:
            with open(path, 'rb') as f:
                f.seek(offset)
                remaining = CHUNK_SIZE
                for i in range(1024):
                    n = f.readinto(buf)
                    if not n:
                        break
                    remaining -= n
                    if remaining <= 0: break
        except: pass

    with ThreadPoolExecutor(max_workers=threads) as pool:
        for _ in range(times):
            list(pool.map(_read_chunk, chunks))

    elapsed = time.time() - start_time
    print(f'Precaching: loaded {file_count} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

precache(WHEELHOUSE_DIR)
precache(MODEL_DIR)
print('precache done:', WHEELHOUSE_DIR, 'and', MODEL_DIR)


## 3. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 4. Import the bundled source

Put the snapshotted repositories on the path (this process and any child processes).

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

## 5. Start serving

Install vLLM from the offline wheelhouse and serve vrfai/Qwen3.6-27B-FP8 (Duck recipe).

In [ ]:
# Cell 11 v4 — vLLM serving for 27B-FP8 UNDER PYTHON3.12 (paste over cell 11 entirely).
# v4 changes (v82: engine child died at init): (1) server PYTHONPATH is now
# STRICTLY the 3.12 wheelhouse target — inheriting the 3.13 kernel's paths can
# inject ABI-incompatible packages into the 3.12 server proc; (2) the
# LD_LIBRARY_PATH prepend is REMOVED (system CUDA-13 libs could shadow torch's
# bundled NCCL with an incompatible build); (3) on failure the cell now prints
# the engine child's own traceback + GPU/shm diagnostics instead of just the
# parent's tail. Install chain unchanged: 3.12 pip -> ensurepip -> manual
# wheel extraction. Duck's launch args unchanged.
import glob
import json
import os
import shutil
import socket
import subprocess
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

VLLM_SITE_PACKAGES = WORKING_DIR / 'vllm-site-packages'
VLLM_SERVER_LOG = WORKING_DIR / 'vllm-openai-server.log'
VLLM_SERVER_PID = WORKING_DIR / 'vllm-openai-server.pid'
VLLM_MAX_MODEL_LEN = 65536
VLLM_BASE_URL = f'http://127.0.0.1:{SERVED_MODEL_PORT}/v1'

print('wheelhouse:', WHEELHOUSE_DIR, flush=True)
print('model:', MODEL_DIR, flush=True)
print('served name:', SERVED_MODEL_NAME, flush=True)

# --- GPU guardrail: Duck's stack targets the RTX Pro 6000. ---
assert shutil.which('nvidia-smi'), 'nvidia-smi missing; GPU box required.'
_smi = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                      capture_output=True, text=True)
assert _smi.returncode == 0, f'nvidia-smi failed: {_smi.stderr.strip()}'
_gpu_names = [l.strip() for l in _smi.stdout.splitlines() if l.strip()]
assert _gpu_names and all('rtx pro 6000' in n.lower() for n in _gpu_names), \
    f'Expected RTX Pro 6000, found {_gpu_names}'
print('GPU check passed:', _gpu_names, flush=True)

# --- verify attached datasets ---
_missing = [p for p in (WHEELHOUSE_DIR, MODEL_DIR) if not Path(p).exists()]
assert not _missing, f'Missing attached dataset path(s): {_missing}.'
_lock = Path(WHEELHOUSE_DIR) / 'requirements.lock'
assert _lock.exists(), f'Missing wheelhouse lock file: {_lock}'

# --- locate python3.12 (cp312-pinned wheelhouse; 3.13 pip cannot use it) ---
def _find_py312():
    for c in (shutil.which('python3.12'), '/usr/bin/python3.12',
              '/usr/local/bin/python3.12'):
        if c and Path(c).exists():
            return c
    for pat in ('/usr/bin/python3.12*', '/usr/local/bin/python3.12*',
                '/opt/*/bin/python3.12'):
        for c in sorted(glob.glob(pat)):
            if Path(c).is_file() and os.access(c, os.X_OK) and 'config' not in c:
                return c
    return None

PY312 = _find_py312()
assert PY312, ('python3.12 not found on this image; the vLLM wheelhouse is '
               'cp312-only. Fix B: re-run this notebook from a fresh '
               'stock-image notebook instead of this custom-image one.')
_v = subprocess.run([PY312, '--version'], capture_output=True, text=True)
assert _v.returncode == 0 and '3.12' in (_v.stdout + _v.stderr), \
    f'bad 3.12 interpreter: {PY312}'
print('server interpreter:', PY312, (_v.stdout + _v.stderr).strip(), flush=True)

# --- refuse a second server on an occupied port ---
with socket.socket() as _probe:
    assert _probe.connect_ex(('127.0.0.1', int(SERVED_MODEL_PORT))) != 0, \
        f'Port {SERVED_MODEL_PORT} is occupied; stop the existing server before rerunning.'

# --- install the wheelhouse for 3.12 (pip -> ensurepip -> manual extract) ---
def _has_pip():
    r = subprocess.run([PY312, '-m', 'pip', '--version'],
                       capture_output=True, text=True)
    return r.returncode == 0

def _pip_install():
    _cmd = [PY312, '-m', 'pip', 'install', '--no-index',
            '--find-links', str(WHEELHOUSE_DIR), '--requirement',
            str(_lock), '--target', str(VLLM_SITE_PACKAGES),
            '--upgrade', '--ignore-installed', '--only-binary', ':all:',
            '--no-compile', '--disable-pip-version-check',
            '--no-warn-conflicts']
    subprocess.run(_cmd, check=True)

def _manual_install():
    # No pip at all: a wheel is a zip, and the lock pins a complete closed
    # set, so plain extraction replicates `pip install --target --no-deps`
    # closely enough for `python -m` serving (dist-info preserved, scripts
    # skipped — the server needs no console entry points).
    _whls = sorted(Path(WHEELHOUSE_DIR).glob('*.whl'))
    assert _whls, f'no wheels found in {WHEELHOUSE_DIR}'
    print(f'manual install: extracting {len(_whls)} wheels with stdlib zipfile',
          flush=True)
    def _target(name):
        p = name.split('/')
        if len(p) > 1 and p[0].endswith('.data'):
            if p[1] in ('purelib', 'platlib'):
                return '/'.join(p[2:]) or None
            return None
        return name
    for _wf in _whls:
        with zipfile.ZipFile(_wf) as _z:
            for _m in _z.namelist():
                if _m.endswith('/'):
                    continue
                _t = _target(_m)
                if not _t:
                    continue
                _dest = VLLM_SITE_PACKAGES / _t
                _dest.parent.mkdir(parents=True, exist_ok=True)
                _dest.write_bytes(_z.read(_m))
    print('manual install: extraction complete', flush=True)

shutil.rmtree(VLLM_SITE_PACKAGES, ignore_errors=True)
VLLM_SITE_PACKAGES.mkdir(parents=True, exist_ok=True)
if _has_pip():
    print('installer: python3.12 pip', flush=True)
    _pip_install()
else:
    print('3.12 has no pip module; trying ensurepip (stdlib)...', flush=True)
    _e = subprocess.run([PY312, '-m', 'ensurepip', '--default-pip'],
                        capture_output=True, text=True)
    print(('ensurepip: ' + (_e.stdout + _e.stderr).strip())[-600:], flush=True)
    if _has_pip():
        print('installer: python3.12 pip (via ensurepip)', flush=True)
        _pip_install()
    else:
        print('installer: manual wheel extraction (no pip, no new datasets)',
              flush=True)
        _manual_install()

# --- verify the install imports under 3.12 before paying for a server boot ---
_chk = subprocess.run(
    [PY312, '-c', 'import vllm, torch; print("vllm", vllm.__version__, "| torch", torch.__version__)'],
    capture_output=True, text=True,
    env={**os.environ, 'PYTHONPATH': str(VLLM_SITE_PACKAGES)})
print('import check:', (_chk.stdout + _chk.stderr).strip()[-600:], flush=True)
assert _chk.returncode == 0, 'vllm/torch do not import under 3.12; see import check output.'

# --- launch under 3.12 (Duck args + env, verbatim; isolated PYTHONPATH) ---
_server_env = dict(os.environ)
_server_env.update({
    'PYTHONPATH': str(VLLM_SITE_PACKAGES),
    'USE_TF': '0', 'TRANSFORMERS_NO_TF': '1', 'TRANSFORMERS_NO_TORCHVISION': '1',
    'VLLM_NO_USAGE_STATS': '1',
})
_launch_cmd = [PY312, '-m', 'vllm.entrypoints.openai.api_server',
               '--model', str(MODEL_DIR), '--served-model-name', SERVED_MODEL_NAME,
               '--host', '127.0.0.1', '--port', str(SERVED_MODEL_PORT),
               '--tensor-parallel-size', '1', '--enable-auto-tool-choice',
               '--tool-call-parser', 'qwen3_coder', '--generation-config', 'vllm',
               '--enable-prefix-caching', '--default-chat-template-kwargs',
               '{"preserve_thinking": true}', '--reasoning-parser', 'qwen3',
               '--max-model-len', str(VLLM_MAX_MODEL_LEN)]

# --- pre-launch diagnostics (GPU mem, shm, server env) ---
_mem = subprocess.run(['nvidia-smi', '--query-gpu=memory.total,memory.free',
                       '--format=csv,noheader'], capture_output=True, text=True)
print('gpu mem (total, free):', (_mem.stdout or _mem.stderr).strip(), flush=True)
_shm = subprocess.run(['df', '-h', '/dev/shm'], capture_output=True, text=True)
print('shm:', (_shm.stdout or _shm.stderr).strip().replace('\n', ' | '), flush=True)
print('server PYTHONPATH:', _server_env['PYTHONPATH'], flush=True)
print('server LD_LIBRARY_PATH:', _server_env.get('LD_LIBRARY_PATH', '<unset>'), flush=True)
print('proxy env:', {k: v for k, v in os.environ.items()
                     if 'proxy' in k.lower()} or '<none>', flush=True)

print('Launching vLLM:', ' '.join(_launch_cmd), flush=True)
VLLM_SERVER_PID.unlink(missing_ok=True)
with open(VLLM_SERVER_LOG, 'w', encoding='utf-8') as _logf:
    _proc = subprocess.Popen(_launch_cmd, env=_server_env, stdout=_logf,
                             stderr=subprocess.STDOUT, text=True,
                             start_new_session=True)
VLLM_SERVER_PID.write_text(str(_proc.pid), encoding='utf-8')
print(f'pid {_proc.pid} -> {VLLM_SERVER_LOG}', flush=True)

# --- wait for readiness (poll /v1/models, 15 min) ---
def _failure_report():
    _rep = []
    try:
        _free = subprocess.run(['nvidia-smi', '--query-gpu=memory.free',
                                '--format=csv,noheader'],
                               capture_output=True, text=True, timeout=30)
        _rep.append('gpu free at failure: ' + (_free.stdout or _free.stderr).strip())
    except Exception as e:
        _rep.append(f'gpu query failed: {e}')
    if not VLLM_SERVER_LOG.exists():
        return '\n'.join(_rep + ['no server log file'])
    _lines = VLLM_SERVER_LOG.read_text(encoding='utf-8',
                                       errors='replace').splitlines()
    _tb = [i for i, l in enumerate(_lines)
           if 'Traceback (most recent call last)' in l]
    if _tb:
        _rep.append(f'--- first traceback at server-log line {_tb[0]+1} ---')
        _rep.extend(_lines[_tb[0]:_tb[0]+50])
    else:
        _rep.append('--- no traceback in server log (child may have been SIGKILLed: OOM?) ---')
    _errs = [f'{i+1}: {l}' for i, l in enumerate(_lines)
             if 'error' in l.lower()][-25:]
    if _errs:
        _rep.append('--- error lines ---')
        _rep.extend(_errs)
    _rep.append('--- log tail ---')
    _rep.extend(_lines[-30:])
    return '\n'.join(_rep)

_t0 = time.time()
while time.time() - _t0 < 900:
    if _proc.poll() is not None:
        raise RuntimeError(f'vLLM server exited ({_proc.returncode}). Failure report:\n{_failure_report()}')
    try:
        with urllib.request.urlopen(VLLM_BASE_URL + '/models', timeout=5) as _r:
            print('READY after', int(time.time() - _t0), 's ->', VLLM_BASE_URL, flush=True)
            print('models:', _r.read().decode()[:300], flush=True)
            break
    except Exception:
        time.sleep(5)
else:
    raise TimeoutError(f'vLLM not ready in 900s. Failure report:\n{_failure_report()}')

# --- smoke test: real model output (Duck prompt, verbatim) ---
_smoke = {'model': SERVED_MODEL_NAME,
          'messages': [{'role': 'user', 'content': 'Answer in one short sentence: what is 2 + 2?'}],
          'temperature': 0.0, 'max_tokens': 96,
          'chat_template_kwargs': {'enable_thinking': False}}
_req = urllib.request.Request(VLLM_BASE_URL + '/chat/completions',
                              data=json.dumps(_smoke).encode(),
                              headers={'Content-Type': 'application/json'})
with urllib.request.urlopen(_req, timeout=300) as _r:
    _gen = json.loads(_r.read().decode())['choices'][0]['message'].get('content', '').strip()
print('=' * 60, flush=True)
print('SMOKE OUTPUT:', _gen, flush=True)
print('=' * 60, flush=True)
assert _gen, 'vLLM smoke test returned empty output.'

# --- agent endpoints must match cell 3's SERVER_BASE_URL ---
os.environ['LOCAL_ANALYZER_BASE_URL'] = SERVER_BASE_URL
os.environ['OPENAI_BASE_URL'] = SERVER_BASE_URL
assert SERVER_BASE_URL == VLLM_BASE_URL, (SERVER_BASE_URL, VLLM_BASE_URL)
print('serving ready: 27B FP8 on', VLLM_BASE_URL, flush=True)

## 6. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

## 7. Customization hook

Optional: tweak `bm`, `bm.games`, or `bm.solver` here before the run starts — the safe place
for one-off experiments once the deployed bundle has loaded.

In [ ]:
# Make one-off changes to `bm`, `bm.games`, or `bm.solver` here before the run starts.
bm.solver.analyzer_timeout = 900.0
bm.solver.max_actions_per_game = None
bm.solver.save_request_logs = True
bm.n_passes = 1
bm.game_weights = None

demo_excluded_games = []
_SMOKE_GAME_PREFIX = 'vc33'  # '' = all 25 bundled games (full demo)

if TRUE_SUBMISSION:  # overrides for contest run
    bm.solver.save_request_logs = False
    bm.n_passes = 1
    if USE_PRIORITY_SCHEDULING:
        bm.solver.concurrency = 120  # game count should be 110
        bm.solver.max_runtime_s_per_game = 532*60
else:
    bm.solver.concurrency = 1
    bm.solver.max_runtime_s_per_game = 20*60
    if float(getattr(target, 'max_runtime_s', 0.0) or 0.0) != 32400.0:
        raise RuntimeError(
            f'Expected the 32400-second notebook budget, got {target.max_runtime_s!r}.'
        )

# show config
print('benchmark.solver:', bm.solver)
if TRUE_SUBMISSION:
    print('SUBMISSION: live competition games via gateway; concurrency=120.')
else:
    print('SMOKE (0.1.21 + Duck-27B): vc33 only; 1200 seconds; concurrency=1.')


## 8. Run the system monitor

Periodically show free main memory, VRAM and disk space.

In [ ]:
# --- system census, one line every 10 minutes -------------------------------
# Everything reads as used/total. RAM is total - MemAvailable, which runs higher
# than the "used" column in `free -h`: that one excludes reclaimable page cache,
# and MemAvailable is the figure that actually predicts an OOM. Disk likewise
# counts the root reserve as unavailable, matching df.
#
# A daemon thread rather than a hook in the run loop: the point of watching a
# long run is to see what happens when it stops behaving, and a print driven by
# the harness stops with it. This keeps reporting through a deadlock.
import os, subprocess, threading, sys

def _sysline(paths=("/", "/kaggle/working")):
    GB = 1024 ** 3
    out = []
    mem = {}
    try:
        for line in open("/proc/meminfo"):
            k, _, rest = line.partition(":")
            p = rest.split()
            if p:
                mem[k] = int(p[0]) * 1024
    except OSError:
        pass
    if mem.get("MemTotal") and mem.get("MemAvailable") is not None:
        out.append(f"RAM {(mem['MemTotal']-mem['MemAvailable'])/GB:.1f}/{mem['MemTotal']/GB:.1f}")
    if mem.get("SwapTotal"):
        out.append(f"swap {(mem['SwapTotal']-mem['SwapFree'])/GB:.1f}/{mem['SwapTotal']/GB:.1f}")
    else:
        out.append("swap none")
    st = {}
    try:
        for line in open("/proc/self/status"):
            k, _, rest = line.partition(":")
            p = rest.split()
            if k in ("VmRSS", "Threads") and p:
                st[k] = int(p[0])
    except OSError:
        pass
    proc = []
    if "VmRSS" in st:
        proc.append(f"{st['VmRSS']*1024/GB:.1f} GB")
    proc.append(f"{st.get('Threads', threading.active_count())} threads")
    try:
        proc.append(f"{len(os.listdir('/proc/self/fd'))} fds")
    except OSError:
        pass
    out.append("proc " + ", ".join(proc))
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total",
                            "--format=csv,noheader,nounits"],
                           capture_output=True, text=True, timeout=5, check=False)
        gpus = []
        for i, line in enumerate(r.stdout.splitlines()):
            p = [x.strip() for x in line.split(",")]
            if len(p) == 2:
                gpus.append(f"GPU{i} {float(p[0])/1024:.1f}/{float(p[1])/1024:.1f}")
        out.append(" ".join(gpus) if gpus else "GPU unavailable")
    except Exception:
        out.append("GPU unavailable")          # nvidia-smi hangs on a wedged GPU
    disks = []
    for path in paths:
        try:
            s = os.statvfs(path)
        except OSError:
            continue
        total = s.f_blocks * s.f_frsize
        disks.append(f"{path} {(total - s.f_bavail*s.f_frsize)/GB:.1f}/{total/GB:.1f}")
    if disks:
        out.append("disk " + ", ".join(disks))
    return "[sys] " + " | ".join(out)


def start_monitor(interval_minutes=10.0, paths=("/", "/kaggle/working")):
    stop = threading.Event()
    def loop():
        while not stop.is_set():
            try:
                print(_sysline(paths), flush=True)
            except Exception as exc:
                print(f"[sys] census failed: {exc!r}", flush=True)
            stop.wait(interval_minutes * 60.0)
    threading.Thread(target=loop, name="system-stats", daemon=True).start()
    return stop

if not TRUE_SUBMISSION:
    _stop_monitor = start_monitor(2)
    print(_sysline())

## 9. Run the benchmark

In a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`), wait for the Kaggle gateway and
play the **live competition Arcade**. Otherwise — an interactive "Save & Run" — play the
competition's **bundled environment files offline**, with no gateway required, so the notebook
runs end-to-end without a submission. Teardown commands run afterward even if the run raises.

In [ ]:
print('Starting benchmark...')

# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    game_ids = [game_id for game_id in game_ids if not any(game_id.startswith(g) for g in demo_excluded_games)]
    if _SMOKE_GAME_PREFIX:
        game_ids = [game_id for game_id in game_ids if game_id.startswith(_SMOKE_GAME_PREFIX)]
    print('Selected demo games (%d):' % len(game_ids), sorted(game_ids))
    _expected = 1 if _SMOKE_GAME_PREFIX else 25
    if len(game_ids) != _expected:
        raise RuntimeError(f'Expected {_expected} demo environments, found {len(game_ids)}: {sorted(game_ids)!r}.')
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Print the run preamble and persist the launcher's git status for diagnostics.
#print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

if TRUE_SUBMISSION:
    # Real submission: play the live competition Arcade served by the Kaggle gateway.
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    # The gateway boots asynchronously; wait before swapping in its game list.
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    # Interactive run: play the bundled competition environments offline (no gateway).
    # The competition's environment files ship alongside the wheelhouse in the competition dataset.
    competition_env_files = str(Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent / "environment_files")
    bm.games = _offline_games(competition_env_files)

# Outside a real submission, stop ~10 min before the wall-clock budget for a graceful exit.
soft_end = None
if not TRUE_SUBMISSION:
    budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
    if budget > 0:
        soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=budget - min(600.0, budget / 2))

# Play the benchmark; teardown commands run even if the run raises.
await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)
if not TRUE_SUBMISSION:
    # An offline run isn't scored, but Kaggle still expects a submission.parquet output.
    import pandas as pd

    pd.DataFrame(
        [["1_0", "1", True, 1]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    ).to_parquet(WORKING_DIR / "submission.parquet", index=False)

## 10. Show the diagnostics

A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is rendered inline below
(and downloadable from the working directory). You should be able to click around through the links.

In [ ]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")